In [ ]:
# -- Load data --
import pandas as pd
import numpy as np

df = pd.read_csv("Filemaker_CLEAN.csv")

df = df[[
    "uniqname",
    "experience_type",
    "year",
    "term_season"
]]

df = df.dropna()
df["experience_type"] = df["experience_type"].str.strip()
df = df.drop_duplicates()

df.head()

,uniqname,experience_type,year,term_season
0,abniemi,Course,2019,FA
1,abniemi,Course,2019,FA
2,abniemi,Counseling Appointment,2021,WN
3,abniemi,Counseling Appointment,2021,WN
4,acurnow,Event,2017,FA


In [10]:
# -- Sort by time --

season_map = {"WN": 1, "SP": 2, "SU": 3, "FA": 4}
df["season_num"] = df["term_season"].map(season_map)

df = df.sort_values(["uniqname", "year", "season_num"])

# First Touch Analysis

In [11]:
# -- First Touch Type Probabilities --

first_touch = (
    df.sort_values(["uniqname", "year", "season_num"])
      .groupby("uniqname")
      .first()
      .reset_index()
)

first_touch_counts = first_touch["experience_type"].value_counts(normalize=True)

print("First-touch distribution:")
print(first_touch_counts)

First-touch distribution:
experience_type
Event                     0.574030
Course                    0.254065
Counseling Appointment    0.094949
Funding                   0.076956
Name: proportion, dtype: float64


In [12]:
# -- Transition matrix -- 
df["next_exp"] = df.groupby("uniqname")["experience_type"].shift(-1)

transitions = df.dropna(subset=["next_exp"])

transition_counts = (
    transitions.groupby(["experience_type", "next_exp"])
    .size()
    .reset_index(name="count")
)

# Normalize
transition_counts["total"] = transition_counts.groupby("experience_type")["count"].transform("sum")
transition_counts["prob"] = transition_counts["count"] / transition_counts["total"]

transition_counts.sort_values(["experience_type", "prob"], ascending=[True, False]).head(20)

,experience_type,next_exp,count,total,prob
0,Counseling Appointment,Counseling Appointment,32207,33263,0.968253
2,Counseling Appointment,Event,592,33263,0.017798
1,Counseling Appointment,Course,268,33263,0.008057
3,Counseling Appointment,Funding,196,33263,0.005892
5,Course,Course,29764,30851,0.964766
6,Course,Event,545,30851,0.017666
4,Course,Counseling Appointment,395,30851,0.012803
7,Course,Funding,147,30851,0.004765
10,Event,Event,69774,71415,0.977022
9,Event,Course,740,71415,0.010362


In [13]:
# -- Conditional Lift --

# Baseline probability of each type
baseline = df["experience_type"].value_counts(normalize=True)

# Merge baseline
transition_counts["baseline"] = transition_counts["next_exp"].map(baseline)

# Compute lift
transition_counts["lift"] = transition_counts["prob"] / transition_counts["baseline"]

# Sort by strongest effects
lift_table = transition_counts.sort_values("lift", ascending=False)

print(lift_table.head(20))

           experience_type                next_exp  count  total      prob  \
15                 Funding                 Funding  11948  12541  0.952715   
5                   Course                  Course  29764  30851  0.964766   
0   Counseling Appointment  Counseling Appointment  32207  33263  0.968253   
10                   Event                   Event  69774  71415  0.977022   
11                   Event                 Funding    438  71415  0.006133   
3   Counseling Appointment                 Funding    196  33263  0.005892   
12                 Funding  Counseling Appointment    182  12541  0.014512   
4                   Course  Counseling Appointment    395  30851  0.012803   
7                   Course                 Funding    147  30851  0.004765   
9                    Event                  Course    740  71415  0.010362   
14                 Funding                   Event    288  12541  0.022965   
13                 Funding                  Course    123  12541

In [14]:
# -- First Touch --> Next Step Counts--

first_df = df.copy()
first_df["order"] = first_df.groupby("uniqname").cumcount()

first_only = first_df[first_df["order"] == 0]
second_only = first_df[first_df["order"] == 1]

first_second = first_only.merge(
    second_only[["uniqname", "experience_type"]],
    on="uniqname",
    suffixes=("_first", "_second")
)

first_second_counts = (
    first_second.groupby(["experience_type_first", "experience_type_second"])
    .size()
    .reset_index(name="count")
)

first_second_counts

,experience_type_first,experience_type_second,count
0,Counseling Appointment,Counseling Appointment,378
1,Course,Course,814
2,Event,Event,1703
3,Funding,Funding,213


# Who Becomes Highly Engaged?

In [15]:
# -- Define High Engagemnent --
engagement_counts = df.groupby("uniqname").size().rename("total_engagements")

student_df = engagement_counts.reset_index()

threshold = student_df["total_engagements"].quantile(0.75)
student_df["high_engagement"] = (student_df["total_engagements"] >= threshold).astype(int)
student_df.head()

,uniqname,total_engagements,high_engagement
0,EMBOBOE,1,0
1,a_wilkin@yahoo.com,1,0
2,aadarm,4,0
3,aahendrx,72,1
4,aalcocer,361,1


In [16]:
# -- Feature Engineering --

# Did student EVER do each type
type_flags = (
    df.assign(val=1)
      .pivot_table(index="uniqname", columns="experience_type", values="val", aggfunc="max", fill_value=0)
)

# First touch type
first_touch_type = first_touch.set_index("uniqname")["experience_type"]

# Merge
features = student_df.set_index("uniqname").join(type_flags)
features["first_touch"] = first_touch_type

features.head()

,total_engagements,high_engagement,Counseling Appointment,Course,Event,Funding,first_touch
uniqname,,,,,,,
EMBOBOE,1,0,0,1,0,0,Course
a_wilkin@yahoo.com,1,0,0,0,1,0,Event
aadarm,4,0,0,1,1,0,Course
aahendrx,72,1,0,0,1,1,Event
aalcocer,361,1,1,1,1,0,Course


In [17]:
# -- Impact Analysis --
impact_results = {}

for col in type_flags.columns:
    group = features.groupby(col)["high_engagement"].mean()
    impact_results[col] = group

impact_df = pd.DataFrame(impact_results).T
impact_df.columns = ["No", "Yes"]

impact_df["lift"] = impact_df["Yes"] / impact_df["No"]

impact_df.sort_values("lift", ascending=False)

,No,Yes,lift
Counseling Appointment,0.166574,0.701286,4.210053
Event,0.130699,0.344859,2.638572
Funding,0.212702,0.554167,2.605368
Course,0.185172,0.442558,2.389981


In [18]:
# -- First Touch Impact --
first_touch_impact = (
    features.reset_index()
    .groupby("first_touch")["high_engagement"]
    .mean()
    .sort_values(ascending=False)
)

print(first_touch_impact)

first_touch
Counseling Appointment    0.538813
Course                    0.283276
Event                     0.254909
Funding                   0.185915
Name: high_engagement, dtype: float64
